# M6 · 검색 품질 평가
⑧ 지침 PDF에서 문서 ID가 분명한 질문 10개를 먼저 고정했다. 같은 청크와 같은 필터로 상위 5문서를 검색한다. ① 사례는 너무 많아 지침 질문을 덮을 수 있으므로 여기서는 `kosha_guide`만 대상으로 한다. 이 점수는 전체 RAG 품질 점수가 아니다.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import chromadb
from sklearn.feature_extraction.text import HashingVectorizer
from IPython.display import display
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
P=ROOT/'data'/'personal'; E=P/'evaluation'; E.mkdir(parents=True,exist_ok=True)
meta=json.loads((P/'corpus'/'index_meta.json').read_text(encoding='utf-8'))
collection=chromadb.PersistentClient(path=str(ROOT/'chroma_db'/'personal')).get_collection(meta['collection'],embedding_function=None)
vectorizer=HashingVectorizer(analyzer='char',ngram_range=(2,4),n_features=1024,alternate_sign=False,norm='l2',dtype=np.float32)
def search(query,k=5,source_type='kosha_guide'):
    vector=vectorizer.transform([query]).toarray().astype(np.float32)
    found=collection.query(query_embeddings=vector.tolist(),n_results=min(100,collection.count()),
                           where={'source_type':source_type},include=['metadatas','distances'])
    hits=[];seen=set()
    for doc,distance in zip(found['metadatas'][0],found['distances'][0]):
        if doc['doc_id'] in seen:continue
        hits.append({'doc_id':doc['doc_id'],'score':round(1-float(distance),4)})
        seen.add(doc['doc_id'])
        if len(hits)==k:break
    return hits
def measure(rows):
    details=[]
    for row in rows:
        hits=search(row['query'],source_type=row['source_type'])
        ranked=[h['doc_id'] for h in hits]
        gold=set(row['gold_doc_ids'].split('|'))
        found=[i+1 for i,doc_id in enumerate(ranked) if doc_id in gold]
        details.append({'qid':row['qid'],'query':row['query'],'gold':row['gold_doc_ids'],
                        'top5':'|'.join(ranked),'hit5':int(bool(found)),
                        'precision5':len(found)/5,'rr':1/min(found) if found else 0})
    result=pd.DataFrame(details)
    scores={name:round(float(result[name].mean()),4) for name in ('hit5','precision5','rr')}
    return result,scores

## 질문과 정답 문서 ID 고정

In [ ]:
questions=[{'qid': 'Q01', 'query': '포크리프트 하역할 때 주의할 점', 'gold_doc_ids': 'B-M-11-2025', 'source_type': 'kosha_guide'}, {'qid': 'Q02', 'query': '모바일크레인 작업할 때 위험 요소', 'gold_doc_ids': 'B-M-8-2025', 'source_type': 'kosha_guide'}, {'qid': 'Q03', 'query': '백호 작업 중 사고를 막는 방법', 'gold_doc_ids': 'D-C-4-2025', 'source_type': 'kosha_guide'}, {'qid': 'Q04', 'query': '사다리에서 높은 곳 작업 시 안전수칙', 'gold_doc_ids': 'A-G-4-2025', 'source_type': 'kosha_guide'}, {'qid': 'Q05', 'query': '용접 중 불이 나지 않게 하려면', 'gold_doc_ids': 'A-G-14-2026', 'source_type': 'kosha_guide'}, {'qid': 'Q06', 'query': '비계 조립과 작업 안전', 'gold_doc_ids': 'D-C-7-2026', 'source_type': 'kosha_guide'}, {'qid': 'Q07', 'query': '전기가 흐르는 선로 근처에서 작업', 'gold_doc_ids': 'B-E-11-2026', 'source_type': 'kosha_guide'}, {'qid': 'Q08', 'query': '작업발판이 달린 거푸집 안전', 'gold_doc_ids': 'D-C-8-2026', 'source_type': 'kosha_guide'}, {'qid': 'Q09', 'query': '고소작업대 선정과 관리 방법', 'gold_doc_ids': 'M-155-2023', 'source_type': 'kosha_guide'}, {'qid': 'Q10', 'query': '크레인에 사용하는 줄걸이 와이어로프', 'gold_doc_ids': 'B-M-12-2025', 'source_type': 'kosha_guide'}]
known={x['doc_id'] for x in collection.get(where={'source_type':'kosha_guide'},include=['metadatas'])['metadatas']}
assert all(set(q['gold_doc_ids'].split('|'))<=known for q in questions)
pd.DataFrame(questions).to_csv(E/'questions.csv',index=False,encoding='utf-8-sig')
display(pd.DataFrame(questions))

## Hit@5, Precision@5, MRR 계산

In [ ]:
details,scores=measure(questions)
display(details)
print('기준선',scores)
details.to_csv(E/'baseline_details.csv',index=False,encoding='utf-8-sig')
(E/'baseline_scores.json').write_text(json.dumps({'questions':len(questions),'top_k':5,
    'source_filter':'kosha_guide','scores':scores,'index':meta['collection']},ensure_ascii=False,indent=2),encoding='utf-8')

## 평가 결과와 범위

- 고정된 KOSHA GUIDE 10문항과 정답 문서 ID를 기준으로 의미 검색의 Hit@5 0.60, Precision@5 0.12, 평균 reciprocal rank 0.395를 기록했다. 같은 질문·필터·상위 5개 기준을 사용한 기준선이다.
- Precision@5가 낮다는 것은 상위 결과 다섯 개 중 관련 문서가 적을 수 있다는 뜻이다. Hit@5는 정답 문서가 한 개라도 포함되는지 보는 값이라 답 내용의 정확도를 평가하지 않는다.
- 시험 질문이 10개이고 모두 안전지침에 대한 것이므로 자료 전체와 사용자 질문을 대표하지 않는다. 답변의 사실성, OCR 정밀도, 실제 안전 효과를 이 점수로 주장하지 않는다.

### 다음 단계에서 볼 것

- 실패 질문의 정답 문서가 코퍼스에 들어갔는지, 검색 가능한 청크가 있는지, 검색어와 원문 용어가 다른지 나눠 본다.
- 같은 고정 질문·정답 문서·필터로 개선 전후를 다시 측정한다.
- 평가지표는 검색 순위 비교 도구이며 LLM 응답을 승인하는 안전 인증 점수가 아니다.
